# 2. Ingest – Ventas de vehículos (`car_sales.csv`)
**Capa:** raw ➜ bronze  |  **Destino:** `vehiculos.bronze.car_sales`

- Lectura del CSV desde el container `raw` del Data Lake con **schema explícito**.
- Renombrado de columnas a *snake_case* (la fuente trae espacios y caracteres especiales: `Price ($)`, `Dealer_No `).
- Conversión de la fecha `M/d/yyyy` a tipo `date` y derivación de `sale_year` (columna de partición).
- Columnas de auditoría: `source_file` e `ingestion_date`.

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, to_date, year, current_timestamp

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "vehiculos")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsjgl99")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/car_sales.csv"

### Schema explícito
Los nombres del schema reemplazan a los del encabezado del CSV (por posición), así se limpian los nombres desde la lectura.
La fecha se lee como `string` porque viene en formato `M/d/yyyy` y se convierte después.

In [0]:
car_sales_schema = StructType(fields=[
    StructField("Car_id", StringType(), False),
    StructField("Date", StringType(), True),
    StructField("Customer_Name", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("Annual_Income", IntegerType(), True),
    StructField("Dealer_Name", StringType(), True),
    StructField("Company", StringType(), True),
    StructField("Model", StringType(), True),
    StructField("Engine", StringType(), True),
    StructField("Transmission", StringType(), True),
    StructField("Color", StringType(), True),
    StructField("Price", IntegerType(), True),
    StructField("Dealer_No", StringType(), True),
    StructField("Body_Style", StringType(), True),
    StructField("Phone", StringType(), True),
    StructField("Dealer_Region", StringType(), True)
])

In [0]:
df_car_sales = spark.read \
    .option("header", True) \
    .option("encoding", "UTF-8") \
    .schema(car_sales_schema) \
    .csv(ruta)

In [0]:
display(df_car_sales.limit(10))

### Renombrado, tipado y columnas derivadas

In [0]:
car_sales_renamed_df = df_car_sales.select(
    col("Car_id").alias("car_id"),
    to_date(col("Date"), "M/d/yyyy").alias("sale_date"),
    col("Customer_Name").alias("customer_name"),
    col("Gender").alias("gender"),
    col("Annual_Income").alias("annual_income"),
    col("Dealer_Name").alias("dealer_name"),
    col("Company").alias("company"),
    col("Model").alias("model"),
    col("Engine").alias("engine"),
    col("Transmission").alias("transmission"),
    col("Color").alias("color"),
    col("Price").alias("price"),
    col("Dealer_No").alias("dealer_no"),
    col("Body_Style").alias("body_style"),
    col("Phone").alias("phone"),
    col("Dealer_Region").alias("dealer_region"),
    col("_metadata.file_name").alias("source_file")
)

In [0]:
car_sales_final_df = car_sales_renamed_df \
    .withColumn("sale_year", year(col("sale_date"))) \
    .withColumn("ingestion_date", current_timestamp())

### Validación previa a la carga

In [0]:
total = car_sales_final_df.count()
fechas_invalidas = car_sales_final_df.filter(col("sale_date").isNull()).count()
ids_duplicados = total - car_sales_final_df.select("car_id").distinct().count()

print(f"Registros leídos : {total}")
print(f"Fechas inválidas : {fechas_invalidas}")
print(f"car_id duplicados: {ids_duplicados}")

### Escritura en bronze
`insertInto` inserta por **posición**, por eso se seleccionan las columnas en el mismo orden de la tabla creada en *1.Preparacion_Ambiente*.

In [0]:
columnas_tabla = spark.table(f"{catalogo}.{esquema}.car_sales").columns

car_sales_final_df.select(*columnas_tabla) \
    .write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.car_sales")

In [0]:
display(spark.table(f"{catalogo}.{esquema}.car_sales")
        .groupBy("sale_year").count().orderBy("sale_year"))